# Replicación en HDFS: Tolerancia a fallos en un clúster Hadoop

En este notebook experimentaremos con la **replicación de bloques** en HDFS para entender cómo Hadoop garantiza la disponibilidad de los datos ante fallos de nodos.

### Conceptos clave

- En HDFS, cada fichero se divide en **bloques** (128 MB por defecto).
- Cada bloque se replica en varios DataNodes. El número de copias es el **factor de replicación**.
- Si un DataNode cae, las copias en otros nodos permiten seguir leyendo los datos.
- Si **todas** las copias de un bloque desaparecen, ese bloque se pierde y el fichero queda corrupto.

### Nuestro clúster

```
┌──────────────────────────────────────────────────────────┐
│                     NameNode (master)                    │
│             Metadatos: qué bloques, dónde                │
└──────────┬──────────┬──────────┬──────────┬──────────────┘
           │          │          │          │
     ┌─────▼──┐ ┌─────▼──┐ ┌─────▼──┐ ┌─────▼──┐
     │worker1 │ │worker2 │ │worker3 │ │worker4 │
     │DataNode│ │DataNode│ │DataNode│ │DataNode│
     │ Bloque │ │ Bloque │ │ Bloque │ │ Bloque │
     │  copia │ │  copia │ │  copia │ │  copia │
     └────────┘ └────────┘ └────────┘ └────────┘
```

Tenemos **4 workers** (DataNodes). El factor de replicación por defecto es **4** (una copia por worker).

### Experimentos

1. **Replicación máxima (4):** subimos un fichero replicado en los 4 workers y paramos workers progresivamente → los datos sobreviven hasta quedarnos sin copias.
2. **Replicación mínima (1):** subimos un fichero con una sola copia y paramos el worker que la tiene → **los datos se pierden**.
3. **Redistribución:** al cambiar el factor de replicación, HDFS copia los bloques automáticamente.

---

> **⚠️ Nota:** Para parar y arrancar workers usaremos comandos `docker compose` que deben ejecutarse desde una **terminal del host** (fuera del contenedor Jupyter), en el directorio `hadoop/`.

## 1. Preparación: subir un fichero de prueba a HDFS

Creamos un pequeño fichero de texto y lo subimos a HDFS para nuestros experimentos.

In [ ]:
import os

os.makedirs("data", exist_ok=True)
!wget -qO data/prestamos_bibliotecas_madrid_2024.csv "http://192.168.70.194/prestamos_bibliotecas_madrid_2024.csv"

# Crear directorio en HDFS para las pruebas de replicación
!hadoop fs -mkdir -p replication_test

# Subir el fichero con factor de replicación 4 (máximo en nuestro clúster)
!hadoop fs -put -f data/prestamos_bibliotecas_madrid_2024.csv replication_test/test_rep4.csv
!hadoop fs -setrep 4 replication_test/test_rep4.csv

# Verificar que está subido
!hadoop fs -ls -h replication_test/

## 2. Estado inicial del clúster

Comprobamos que todos los DataNodes están activos y vemos el informe del clúster.

In [ ]:
# Informe del clúster: nodos activos, capacidad, etc.
!hdfs dfsadmin -report

## 3. Inspeccionar la replicación de un fichero

Con `hdfs fsck` podemos ver en detalle cómo está replicado cada bloque de un fichero: en cuántos nodos está y en cuáles exactamente.

In [ ]:
# Ver información detallada de bloques y sus ubicaciones
!hdfs fsck replication_test/test_rep4.csv -files -blocks -locations

Observa:
- **Replication:** el factor de replicación configurado
- **Block locations:** las IPs de los DataNodes que almacenan cada copia del bloque
- Con replicación 4, el bloque debería estar en los 4 workers (172.28.1.3 a 172.28.1.6)

Verifiquemos que el fichero se puede leer sin problemas:

In [ ]:
# Leer el fichero para confirmar que es accesible
!hadoop fs -head replication_test/test_rep4.csv

---

## 4. Experimento 1: Replicación alta (factor 4) — Tolerancia a fallos

Con el fichero replicado en los 4 workers, vamos a parar workers uno a uno y comprobar que el fichero sigue accesible mientras quede **al menos una copia**.

### Paso 4.1: Parar el primer worker

Ejecuta el siguiente comando en una **terminal del host** (en el directorio `hadoop/`):

```bash
docker compose stop worker1
```

Espera unos segundos y luego ejecuta las siguientes celdas para verificar el estado.

In [ ]:
# Verificar cuántos DataNodes siguen activos
!hdfs dfsadmin -report

In [ ]:
# ¿Sigue accesible el fichero? (Quedan 3 copias)
!hadoop fs -head replication_test/test_rep4.csv

In [ ]:
# ¿En qué nodos quedan las copias del bloque?
!hdfs fsck replication_test/test_rep4.csv -files -blocks -locations

### Paso 4.2: Parar el segundo y tercer worker

Ejecuta en la **terminal del host**:

```bash
docker compose stop worker2 worker3
```

In [ ]:
# Verificar cuántos DataNodes siguen activos
!hdfs dfsadmin -report

In [ ]:
# ¿Sigue accesible el fichero? (Queda 1 copia)
!hadoop fs -head replication_test/test_rep4.csv

In [ ]:
# Ubicación de las copias restantes
!hdfs fsck replication_test/test_rep4.csv -files -blocks -locations


### Paso 4.3: Parar el cuarto y último worker

Ejecuta en la **terminal del host**:

```bash
docker compose stop worker4
```

In [ ]:
# Verificar cuántos DataNodes siguen activos
!hdfs dfsadmin -report

In [ ]:
# Intentar leer el fichero. ¡DEBERÍA FALLAR!
!hadoop fs -head replication_test/test_rep4.csv

### Paso 4.4: Restaurar todos los workers

Ejecuta en la **terminal del host**:

```bash
docker compose start worker1 worker2 worker3 worker4
```

Espera unos 15-20 segundos para que los DataNodes se re-registren con el NameNode.

In [ ]:
# Verificar que todos están de vuelta
!hdfs dfsadmin -report

In [ ]:
# Confirmar que el fichero está accesible de nuevo
!hadoop fs -head replication_test/test_rep4.csv

In [ ]:
# Verificar que la replicación vuelve a ser completa
!hdfs fsck replication_test/test_rep4.csv -files -blocks -locations

**Conclusión:** Con replicación 4, podemos perder hasta 3 de los 4 workers y seguir accediendo a los datos. Solo cuando caen **todos** los nodos se pierde el acceso.

---

## 5. Experimento 2: Replicación baja (factor 1) — Sin tolerancia a fallos

Ahora veremos qué pasa cuando guardamos un fichero con **replicación 1**: solo existe **una copia** del bloque en un único DataNode.

In [ ]:
# Subir un fichero con replicación 1
!hadoop fs -D dfs.replication=1 -put -f data/prestamos_bibliotecas_madrid_2024.csv replication_test/test_rep1.csv

# Verificar la replicación (la 2ª columna muestra el factor)
!hadoop fs -ls replication_test/

In [ ]:
# Ver el espacio ocupado por cada archivo
!hadoop fs -du -h replication_test/

In [ ]:
# ¿En qué worker está la ÚNICA copia del bloque?
!hdfs fsck replication_test/test_rep1.csv -files -blocks -locations

Observa la IP del DataNode que tiene la única copia del bloque.

Recuerda la correspondencia de IPs con workers:

| Worker   | IP           |
|----------|-------------|
| worker1  | 172.28.1.3  |
| worker2  | 172.28.1.4  |
| worker3  | 172.28.1.5  |
| worker4  | 172.28.1.6  |

### Paso 5.1: Parar un worker que NO tiene bloques del archivo.

Consulta la salida del `fsck` anterior para ver en qué workers están los bloques. Luego para **un worker diferente**.

Por ejemplo, si no hay bloques en worker4:

```bash
docker compose stop worker4
```

In [ ]:
# ¿Sigue accesible?
!hadoop fs -head replication_test/test_rep1.csv


### Paso 5.2: Parar un worker que TIENE bloques

Ahora para el worker que tenga la copia del primer bloque.

Por ejemplo, si la copia está en worker1:

```bash
docker compose stop worker1
```

In [ ]:
# Intentar leer el fichero
!hadoop fs -head replication_test/test_rep1.csv

In [ ]:
# Verificar el estado del fichero — bloques perdidos (missing)
!hdfs fsck replication_test/test_rep1.csv -files -blocks -locations

Con replicación 1, basta con perder **un solo worker** para que el dato sea irrecuperable. Fíjate en la salida de `fsck`: debería indicar **missing blocks** o **corrupt blocks**.

Mientras tanto, el fichero con replicación 4 sigue accesible (aún quedan copias en los workers activos):

In [ ]:
# El fichero con replicación 4 sigue funcionando 
!hadoop fs -head replication_test/test_rep4.csv

### Paso 5.3: Restaurar los workers

Ejecuta en la **terminal del host**:

```bash
docker compose start worker1 worker2 worker3 worker4
```

Espera unos segundos y comprueba que se recupera todo.

In [ ]:
# Verificar que todos están de vuelta
!hdfs dfsadmin -report 

In [ ]:
# El fichero con rep=1 debería volver a ser accesible (el worker vuelve con sus datos)
!hadoop fs -head replication_test/test_rep1.csv

---

## 6. Experimento 3: Cambiar la replicación y observar la redistribución

HDFS permite cambiar el factor de replicación de un fichero **después** de haberlo subido. Cuando aumentamos la replicación, el NameNode ordena crear copias adicionales. Cuando la reducimos, las copias sobrantes se eliminan gradualmente.

In [ ]:
# Estado actual: el fichero tiene replicación 1
!hdfs fsck replication_test/test_rep1.csv -files -blocks -locations

In [ ]:
# Aumentar la replicación de 1 a 3
!hadoop fs -setrep 3 replication_test/test_rep1.csv

In [ ]:
# Verificar: ahora debería haber 3 copias en 3 workers diferentes
!hdfs fsck replication_test/test_rep1.csv -files -blocks -locations

In [ ]:
# Comparar espacio en disco: test_rep4.csv (rep=4) vs test_rep1.csv (ahora rep=3)
# La primera columna es el tamaño del fichero, la segunda el espacio total consumido (tamaño × replicación)
!hadoop fs -du -h replication_test/

Observa cómo el bloque ahora aparece en **3 DataNodes** distintos. HDFS ha creado automáticamente 2 copias adicionales.

---

## 7. Limpieza

In [ ]:
# Eliminar los ficheros de prueba de HDFS
!hadoop fs -rm -r -f replication_test

---

## 8. Puntos clave

1. **La replicación es la base de la tolerancia a fallos en HDFS.** Sin réplicas, un fallo de hardware implica pérdida de datos.

2. **El factor de replicación se puede ajustar por fichero**, no es una configuración global rígida. Puedes tener ficheros críticos con replicación alta y ficheros temporales con replicación baja.

3. **HDFS re-replica automáticamente** cuando detecta que un bloque está under-replicated. No es necesaria intervención manual.

4. **Más replicación = más uso de disco.** Con replicación 4 y 4 workers, cada fichero ocupa 4 veces su tamaño original en el clúster.

5. **En producción se usa típicamente replicación 3**, que es el valor por defecto de Hadoop. Esto tolera la caída simultánea de 2 nodos.

6. **La replicación NO es un backup.** Si se borra un fichero (`hadoop fs -rm`), se borran todas las réplicas. Para backups se necesitan mecanismos adicionales (snapshots, distcp a otro clúster, etc.).